# Experimento do TCC — Text-to-SQL com recuperação de demonstrações

Este notebook não contém lógica do experimento. Ele clona o repositório,
instala, escreve a configuração e chama a linha de comando — a lógica toda
vive no GitHub, versionada.

## Roteiro de execução

O notebook tem duas partes e **a ordem importa**. A parte 1 roda sem GPU
porque ela não gera texto: ela existe para provar que o encanamento está
ligado antes de você gastar cota de GPU descobrindo que um caminho do Drive
estava errado.

| Seção | Parte 1 · sem GPU | Parte 2 · com T4 |
|---|---|---|
| 1 · Drive e dados | **rode** | **rode de novo** |
| 2 · Clonar o código | **rode** | **rode de novo** |
| 3 · Instalar | **rode** | **rode de novo** |
| 4 · Conferir config de fumaça | **rode** | pule |
| 5 · Pipeline sem GPU | **rode** (3 células) | pule |
| — trocar para T4 — | | |
| 6 · Confirmar a GPU | pule | **rode** |
| 7 · Instalar modelo | pule | **rode** (2 células) |
| 8 · Calibração | pule | **rode** |
| 9 · Investigar | pule | só se precisar |

**Por que repetir 1, 2 e 3 na parte 2:** trocar o tipo de ambiente reinicia a
sessão e apaga o `/content`. O Drive e o que está nele sobrevivem; o
repositório clonado e os pacotes instalados, não.

**Por que pular 4 e 5 na parte 2:** o `work_dir` aponta para o Drive, então o
manifesto, as partições, o índice do Qdrant e o cache de vetores já estão lá
da parte 1. Não precisa refazer. A parte 2 usa `configs/colab.yaml`, que é outro
arquivo e já vem do repositório.

## 1 · Montar o Drive e descompactar os dados

Rode nas duas partes. A descompactação se resolve sozinha: se os dados já estiverem lá, ela é pulada.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import zipfile, pathlib

base = pathlib.Path('/content/drive/MyDrive/tcc')
if (base / 'data/spider/tables.json').exists():
    print('dados já descompactados, nada a fazer')
else:
    with zipfile.ZipFile(base / 'spider_tcc.zip') as z:
        z.extractall(base)
    print('descompactado')

print(sorted(p.name for p in (base / 'data/spider').iterdir()))

A última linha tem que mostrar `database`, `dev.json`, `tables.json` e `train_spider.json`.

## 2 · Trazer o código

Rode nas duas partes.

In [ ]:
%cd /content
!rm -rf TCC2
!git clone -q https://github.com/VitorEduardoLimaKenor/TCC2.git
%cd TCC2
!git log --oneline -1

O `rm -rf` é intencional: toda sessão começa do código que está no GitHub. Você nunca edita código aqui dentro — edita na sua máquina, dá `push`, e roda esta célula de novo.

## 3 · Instalar

Rode nas duas partes.

In [ ]:
!pip install -q -e .
!python -c "import tcc_rag; print('tcc_rag ok')"

---
## 4 · Conferir a configuração do teste de fumaça

**Só na parte 1.**

As configurações vivem no repositório, não no notebook. A seção 2 faz `rm -rf` e reclona a cada sessão, então arquivo criado por `%%writefile` desaparece — e configuração é parte do experimento, tem que ser versionada.

`backend: dummy` não carrega modelo nenhum: esta rodada não mede nada, ela responde a outra pergunta, mais urgente — o encanamento está ligado?

In [ ]:
!cat configs/colab-fumaca.yaml

## 5 · Rodar o pipeline completo, sem GPU

**Só na parte 1.** São três células.

A indexação baixa o modelo de *embeddings* (~1 GB) e vetoriza 2242 demonstrações na CPU — leve alguns minutos. O resultado fica no Drive, então a parte 2 não precisa repetir.

In [ ]:
!tcc-rag prepare   --config configs/colab-fumaca.yaml
!tcc-rag partition --config configs/colab-fumaca.yaml
!tcc-rag leakage   --config configs/colab-fumaca.yaml

In [ ]:
!tcc-rag index --config configs/colab-fumaca.yaml

In [ ]:
!tcc-rag run      --config configs/colab-fumaca.yaml --split val --limit 20
!tcc-rag evaluate --config configs/colab-fumaca.yaml

### O que confirma que passou

- `bancos elegíveis (>= 90 perguntas, >= 25 famílias): 17`
- `VEREDITO: limpo`
- `indexadas 2242 demonstrações`
- cinco linhas `C0:` a `C4:`, cada uma com `+20 linhas`
- a tabela final com a coluna `n` preenchida

**EX igual a zero é o esperado aqui.** O gerador falso devolve `SELECT 1`.
O que importa é que nada quebrou e que o log foi escrito no Drive.

---
---
# PARE AQUI. Parte 2 · Com GPU

Só continue depois que a parte 1 tiver terminado com `VEREDITO: limpo` e as
cinco condições preenchidas.

### O que fazer agora, na ordem

1. **Ambiente de execução → Alterar o tipo de ambiente de execução → GPU T4 → Salvar.**
   A sessão reinicia e o `/content` é apagado.
2. Volte ao topo e rode **as seções 1, 2 e 3** de novo.
   A seção 1 vai imprimir "dados já descompactados" — é o esperado.
3. **Pule as seções 4 e 5.** O manifesto, as partições e o índice do Qdrant
   já estão no Drive, e a parte 2 usa outra configuração, `configs/colab.yaml`.
4. Siga daqui.

## 6 · Confirmar a GPU

**Só na parte 2.**

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

Precisa dizer `Tesla T4, 15360 MiB`. Se disser que não há GPU, a cota do Colab gratuito acabou — espere algumas horas ou use o Kaggle.

## 7 · Instalar o suporte a modelo e reescrever a configuração

**Só na parte 2.** São duas células: a instalação e a configuração.

In [ ]:
!pip install -q -e ".[hf]"
!pip install -q bitsandbytes

In [ ]:
!cat configs/colab.yaml

## 8 · A rodada de calibração

**Só na parte 2.**

Este é o passo que separa *"meu número é baixo"* de *"meu código está
quebrado"*, e ele vem antes de qualquer construção em cima.

Roda C0 (sem demonstrações) em inglês. O primeiro comando baixa o modelo
(~5 GB) — leva alguns minutos na primeira vez.

In [ ]:
!tcc-rag run --config configs/colab.yaml --split val --limit 80 \
             --conditions C0 --langs en
!tcc-rag evaluate --config configs/colab.yaml

### Como ler o resultado

| EX de C0 em inglês | Leitura |
|---|---|
| **0,45 – 0,65** | Coerente. Um 7B em 4 bits abaixo de um 14B é esperado. Siga. |
| **0,25 – 0,45** | Baixo demais para ser só o porte. Quase sempre é a extração do SQL da resposta, ou o modelo respondendo com explicação em vez de consulta. |
| **abaixo de 0,20** | Algo quebrado. Suspeitos, nesta ordem: o DDL não chega ao *prompt*, o avaliador compara errado, ou o caminho do SQLite está errado e toda execução falha. |

Carvalho *et al.* reportaram ~0,63 para o Qwen2.5-Coder-14B em inglês.

## 9 · Investigar, se o número não bater

In [ ]:
import json, collections
from pathlib import Path
from tcc_rag.logging_io import report_rows

artefatos = Path('/content/drive/MyDrive/tcc/artifacts')
selecao = json.loads((artefatos / 'summary_selection.json').read_text(encoding='utf-8'))
linhas = report_rows(artefatos / 'runs', config_hash=selecao['config_hash'], run_ids=selecao['run_ids'], split=selecao['split'], partition_digest=selecao['partition_digest'])
print(selecao['run_ids'], '|', len(linhas), 'linhas')

falhas = collections.Counter(r['exec_error'] for r in linhas if not r['exec_ok'])
print('\nerros de execução mais comuns:')
for erro, n in falhas.most_common(5):
    print(f'  {n:>4}  {erro}')

print('\n--- exemplo de saída bruta do modelo ---')
print(linhas[0]['raw_output'][:600])
print('\n--- SQL extraído ---')
print(linhas[0]['generated_sql'])
print('\n--- SQL de referência ---')
print(linhas[0]['gold_sql'])

---
## 10 · Fase de validação: varredura de k, λ e N

**Só depois que a calibração estiver coerente.**

Determina os parâmetros do *framework* (Seção 4.6.3). Roda **apenas sobre a
validação** — o comando recusa qualquer outro split, porque o teste fica selado
até o congelamento.

O modelo é carregado uma vez para toda a grade, e os rascunhos de C4 são
gerados na primeira combinação e reaproveitados pelas onze seguintes.

Com ~1,6 s por geração, doze combinações sobre 150 itens levam **cerca de
cinquenta minutos**.

In [ ]:
!tcc-rag sweep --config configs/colab.yaml --limit 150 --langs pt \
               --k 1 4 8 --lambdas 0.5 0.7 0.85 0.95 \
               --n-candidates 25 --conditions C4

### Como ler

A tabela sai ordenada por EX. O que interessa não é só qual ganhou, mas o
**formato da curva em λ**:

- se o melhor λ ficar em 0,5 ou 0,7, a estrutura está contribuindo;
- se subir monotonicamente até 0,95, a estrutura está atrapalhando e C4
  converge para C3 — o que é um achado sobre a hipótese H3, não um fracasso;
- se as quatro ficarem dentro de um ou dois pontos, o efeito de λ é menor que
  o ruído nesse tamanho de amostra, e vale registrar isso.

Rode `!tcc-rag sweep` também com `--conditions C2 C3` (uma combinação basta,
já que eles ignoram λ e N) para ter a referência sob os mesmos 150 itens.

---
## Depois disso

Com a calibração coerente, a ordem é: baselines C1, C2 e C3 → método C4 →
varredura de `k`, `n_candidates` e `lambda_question` na validação →
congelar a configuração → teste confirmatório com `--split test`.

O `run` é retomável: se a sessão cair, rode a mesma célula de novo que ele
pula o que já está no log.